In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

In [ ]:
def quick_prep(gdf):
    gdf = gdf.to_crs('ESRI:102033')
    gdf['geometry'] = gdf.make_valid()
    gdf = gdf.loc[gdf.is_valid]
    gdf['area_ha'] = gdf.geometry.area/10000
    gdf['id'] = np.arange(gdf.shape[0])
    return gdf

In [ ]:
res_gdf = gpd.read_file('../clean_summarize/out_polys/sentinel_2021_v7_aea_cleaned_no0.gpkg')
res_gdf = quick_prep(res_gdf)
res_gdf = res_gdf.loc[res_gdf['area_ha']>=0.05]
res_gdf = res_gdf.loc[res_gdf['area_ha']<=50]
res_gdf['source'] = 'resid'

In [ ]:
ana_gdf = gpd.read_file('../compare_previous_results/data/ana/Massas_d_Agua.shp')
ana_gdf = ana_gdf.loc[ana_gdf['detipomass']=='Artificial']
ana_gdf = quick_prep(ana_gdf)
ana_gdf['source'] = 'ana'

In [ ]:
gdw_gdf = gpd.read_file('../compare_previous_results/data/gdw/GDW_v1_0_shp/GDW_reservoirs_v1_0.shp')
gdw_gdf = gdw_gdf.loc[gdw_gdf['COUNTRY']=='Brazil']
gdw_gdf = quick_prep(gdw_gdf)
gdw_gdf['source'] = 'gdw'

In [ ]:

car_gdf = gpd.read_file('../compare_previous_results/data/car/full_reservoirs_dissolve_explode.shp')
car_gdf = quick_prep(car_gdf)
car_gdf['source'] = 'car'

In [ ]:
mb_gdf = gpd.read_file('../compare_previous_results/data/mapbiomas/collection4_v1.gpkg')
mb_gdf = mb_gdf.loc[mb_gdf['DN']==3]
mb_gdf = quick_prep(mb_gdf)
mb_gdf['source'] = 'mb'

In [ ]:
macedo_gdf = gpd.read_file('../compare_previous_results/data/macedo_2013/aster_reservoirs_masked_aug062011.shp')
macedo_gdf = quick_prep(macedo_gdf)
macedo_gdf['source'] = 'macedo'

In [ ]:

lat_gdf = gpd.read_file('../compare_previous_results/data/lat_2019/v5/jordan_clipped/mg_ls_2014_clipped_class2pos200m_acut1000_mosaic_aea.gpkg')
lat_gdf = lat_gdf.loc[lat_gdf['DN']==1]
lat_gdf = quick_prep(lat_gdf)
lat_gdf['source'] = 'lat'

In [ ]:
# For each compared to reservoir ID: 
# 1. Total number and area intersection
# 2. Total number and area of union
def intersection_union(df1, df2):
    df2 = df2.loc[df2['area_ha']<=50].copy()
    df_intersect = df1.sjoin(df2, how='inner')
    df_intersect = df_intersect.groupby('id_right').first().reset_index()
    df_intersect['area_max'] = df_intersect[['area_ha_left', 'area_ha_right']].max(axis=1)
    df1_unique = df1.loc[~df1['id'].isin(df_intersect['id_left'])]
    df2_unique = df2.loc[~df2['id'].isin(df_intersect['id_right'])]
    out_dict = {
        'name': df1['source'].values[0] + '-' + df2['source'].values[0],
        'left_area': df1['area_ha'].sum(),
        'left_count': df1.shape[0],
        'right_area': df2['area_ha'].sum(),
        'right_count': df2.shape[0],
        'count_intersect': df_intersect.shape[0],
        'left_area_intersect': df_intersect['area_ha_left'].sum(),
        'right_area_intersect': df_intersect['area_ha_right'].sum(),
        'union_count': df_intersect.shape[0] + df1_unique.shape[0] + df2_unique.shape[0],
        'union_area': df_intersect['area_max'].sum() + df1_unique['area_ha'].sum() + df2_unique['area_ha'].sum(),
        'left_area_union': df_intersect['area_ha_left'].sum() + df1_unique['area_ha'].sum() + df2_unique['area_ha'].sum(),
        'right_area_union': df_intersect['area_ha_right'].sum() + df2_unique['area_ha'].sum() + df1_unique['area_ha'].sum(),
    }
    return out_dict


In [ ]:
out_dicts = []
out_dicts.append(intersection_union(res_gdf, gdw_gdf))
out_dicts.append(intersection_union(res_gdf, ana_gdf))
out_dicts.append(intersection_union(res_gdf, car_gdf))
out_dicts.append(intersection_union(res_gdf, mb_gdf))
out_dicts.append(intersection_union(res_gdf, macedo_gdf))
out_dicts.append(intersection_union(res_gdf, lat_gdf))

In [ ]:
pd.DataFrame(out_dicts)

In [ ]:
# Minimum check: *One* of the other datasets must agree
all_indices = []
for compare in [gdw_gdf, car_gdf, mb_gdf, ana_gdf]:
    all_indices.append(res_gdf.sjoin(compare).index.values)
res_minimum = res_gdf.loc[np.unique(np.concatenate(all_indices))]


In [ ]:
res_minimum = res_gdf.loc[np.unique(np.concatenate(all_indices))]
print(res_minimum.shape)
print(res_minimum['area_ha'].sum())

In [ ]:
def quick_union(df1, df2):
    df1 = df1[['area_ha','geometry', 'id']].copy()
    df2 = df2[['area_ha','geometry', 'id']].copy()
    df_intersect = df1.sjoin(df2, how='inner').fillna(0)
    df_intersect = df_intersect.groupby('id_right').first().reset_index()
    df_intersect['area_ha'] = df_intersect[['area_ha_left', 'area_ha_right']].max(axis=1)
    df1_unique = df1.loc[~df1['id'].isin(df_intersect['id_left'])]
    df2_unique = df2.loc[~df2['id'].isin(df_intersect['id_right'])]
    df_union = pd.concat([df_intersect, df1_unique, df2_unique], axis=0, ignore_index=True)
    df_union['id'] = np.arange(df_union.shape[0])
    return df_union.drop(columns=['index_right', 'id_left','id_right', 'area_ha_left','area_ha_right']).reset_index().drop(columns=['index'])

In [ ]:
full_gdf = pd.concat([res_gdf, gdw_gdf, ana_gdf, car_gdf, lat_gdf, macedo_gdf])

In [ ]:
full_gdf_dissolve_explode = full_gdf.dissolve().explode()

In [ ]:
# Absolute maximum: get union of all datasets, taking maximum area of each
temp_gdf = quick_union(res_gdf, gdw_gdf)
temp_gdf = quick_union(temp_gdf, ana_gdf)
temp_gdf = quick_union(temp_gdf, mb_gdf)
temp_gdf = quick_union(temp_gdf, car_gdf)
temp_gdf = quick_union(temp_gdf, lat_gdf)
temp_gdf = quick_union(temp_gdf, macedo_gdf)

In [ ]:
temp_gdf.shape

In [ ]:
macedo_gdf['area_ha'].sum()/100

In [ ]:

ana_gdf['area_ha'].sum()/100

In [ ]:
mb_gdf['area_ha'].sum()/100

In [ ]:
res_gdf['area_ha'].sum()/100

In [ ]:
gdw_gdf['area_ha'].sum()/100

In [ ]:
lat_gdf['area_ha'].sum()/100

In [ ]:
temp_gdf['area_ha'].sum()